# 21 — Broadcast vs Shuffle Join Pattern

Purpose: choose correct join strategy.

Process:

SMALL TABLE + BIG TABLE
  |
  +--> broadcast join (no shuffle)
  |
  +--> shuffle join (expensive)

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("broadcast-vs-shuffle")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions","8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Step 1 — Input

In [2]:
big = spark.createDataFrame(
    [(f"e{i}", f"u{i%100}", i) for i in range(1000)],
    ["event_id","user_id","amount"]
)

small = spark.createDataFrame(
    [(f"u{i}", f"country{i%3}") for i in range(100)],
    ["user_id","country"]
)

print("big:", big.count())
print("small:", small.count())

big: 1000
small: 100


## Step 2 — Normal join (shuffle)

In [3]:
normal = big.join(small, "user_id")

normal.explain()
normal.count()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#1, event_id#0, amount#2L, country#4]
   +- SortMergeJoin [user_id#1], [user_id#3], Inner
      :- Sort [user_id#1 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(user_id#1, 8), ENSURE_REQUIREMENTS, [plan_id=93]
      :     +- Filter isnotnull(user_id#1)
      :        +- Scan ExistingRDD[event_id#0,user_id#1,amount#2L]
      +- Sort [user_id#3 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(user_id#3, 8), ENSURE_REQUIREMENTS, [plan_id=94]
            +- Filter isnotnull(user_id#3)
               +- Scan ExistingRDD[user_id#3,country#4]




1000

## Step 3 — Broadcast join

In [4]:
broadcasted = big.join(F.broadcast(small), "user_id")

broadcasted.explain()
broadcasted.count()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#1, event_id#0, amount#2L, country#4]
   +- BroadcastHashJoin [user_id#1], [user_id#3], Inner, BuildRight, false
      :- Filter isnotnull(user_id#1)
      :  +- Scan ExistingRDD[event_id#0,user_id#1,amount#2L]
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=328]
         +- Filter isnotnull(user_id#3)
            +- Scan ExistingRDD[user_id#3,country#4]




1000

## Step 4 — Rule

In [5]:
print("Use broadcast when one table is small enough to fit memory")

Use broadcast when one table is small enough to fit memory


In [6]:
spark.stop()